In [8]:
import os
import requests
from datetime import datetime, date
import time
from dotenv import load_dotenv

# ---------------------------
# ENV
# ---------------------------
load_dotenv()

POLYGON_API_KEY = os.getenv("POLYGON_API_KEY")
if not POLYGON_API_KEY:
    raise EnvironmentError("Falta POLYGON_API_KEY")

# ---------------------------
# Configuración filtros opciones
# ---------------------------
MIN_DTE = 8
MAX_DTE = 65
MIN_DELTA = -0.35
MAX_DELTA = -0.25
CONTRACT_TYPE = "put"

# ---------------------------
# Lista de tickers (mock / input manual)
# ---------------------------
tickers_validos = ["SCCO"]  # podés poner varios

print(f"Procesando {len(tickers_validos)} tickers...\n")

# ---------------------------
# Consulta Polygon
# ---------------------------
def buscar_opciones_por_ticker(ticker_raiz):
    hoy = datetime.now()
    url = f"https://api.polygon.io/v3/snapshot/options/{ticker_raiz}?limit=250&apiKey={POLYGON_API_KEY}"

    resultados = []
    buscando = True

    while url and buscando:
        r = requests.get(url)
        if r.status_code != 200:
            print(f"❌ Error Polygon {ticker_raiz}: {r.status_code}")
            break

        data = r.json()
        contracts = data.get("results", [])

        for c in contracts:
            details = c.get("details", {})
            greeks = c.get("greeks", {})
            day = c.get("day", {})

            if details.get("contract_type") != CONTRACT_TYPE:
                continue

            expiry = details.get("expiration_date")
            if not expiry:
                continue

            fecha_vto = datetime.strptime(expiry, "%Y-%m-%d")
            dte = (fecha_vto - hoy).days

            # Corte temprano (Polygon viene ordenado por vto)
            if dte > MAX_DTE:
                buscando = False
                break

            delta = greeks.get("delta")
            if delta is None or not (MIN_DELTA <= delta <= MAX_DELTA):
                continue

            if MIN_DTE <= dte <= MAX_DTE:
                resultados.append({
                    "opcion": details.get("ticker"),
                    "strike": details.get("strike_price"),
                    "vto": expiry,
                    "dte": dte,
                    "delta": round(delta, 4),
                    "iv": c.get("implied_volatility"),
                    "oi": c.get("open_interest", 0),
                    "volume": day.get("volume", 0),
                    "close": day.get("close")
                })

        next_url = data.get("next_url")
        url = f"{next_url}&apiKey={POLYGON_API_KEY}" if next_url else None

    return resultados

# ---------------------------
# Ejecutar y mostrar
# ---------------------------
total_opciones = 0

for ticker in tickers_validos:
    print(f"📌 TICKER: {ticker}")
    print("=" * 80)

    opciones = buscar_opciones_por_ticker(ticker)

    if not opciones:
        print("No se encontraron PUTs válidos\n")
        continue

    for o in opciones:
        print(
            f"{o['opcion']:20} | "
            f"Vto {o['vto']} | "
            f"Strk {o['strike']:>6} | "
            f"DTE {o['dte']:>3} | "
            f"Δ {o['delta']:>6} | "
            f"IV {o['iv']:.3f} | "
            f"OI {o['oi']:>5} | "
            f"Vol {o['volume']:>5}"
        )

    print(f"\n➡️ Total opciones válidas: {len(opciones)}\n")
    total_opciones += len(opciones)

    time.sleep(0.25)

print(f"✅ Proceso finalizado. Total opciones encontradas: {total_opciones}")


Procesando 1 tickers...

📌 TICKER: SCCO
O:SCCO260424P00180000 | Vto 2026-04-24 | Strk    180 | DTE   8 | Δ -0.2695 | IV 0.491 | OI    39 | Vol     3
O:SCCO260501P00175000 | Vto 2026-05-01 | Strk    175 | DTE  15 | Δ -0.2612 | IV 0.604 | OI     4 | Vol     1
O:SCCO260501P00177500 | Vto 2026-05-01 | Strk  177.5 | DTE  15 | Δ -0.2984 | IV 0.604 | OI     3 | Vol     1
O:SCCO260501P00180000 | Vto 2026-05-01 | Strk    180 | DTE  15 | Δ -0.3356 | IV 0.598 | OI     4 | Vol     4
O:SCCO260508P00172500 | Vto 2026-05-08 | Strk  172.5 | DTE  22 | Δ -0.2602 | IV 0.619 | OI     0 | Vol     0
O:SCCO260508P00175000 | Vto 2026-05-08 | Strk    175 | DTE  22 | Δ -0.2955 | IV 0.642 | OI     5 | Vol     1
O:SCCO260508P00177500 | Vto 2026-05-08 | Strk  177.5 | DTE  22 | Δ -0.3238 | IV 0.632 | OI    18 | Vol     6
O:SCCO260508P00180000 | Vto 2026-05-08 | Strk    180 | DTE  22 | Δ -0.3463 | IV 0.559 | OI    14 | Vol     1
O:SCCO260515P00175000 | Vto 2026-05-15 | Strk    175 | DTE  29 | Δ -0.3008 | IV 0.591 | 

In [3]:
import os
import requests
from datetime import datetime, date
import time
from dotenv import load_dotenv

# ---------------------------
# ENV
# ---------------------------
load_dotenv()

POLYGON_API_KEY = os.getenv("POLYGON_API_KEY")
if not POLYGON_API_KEY:
    raise EnvironmentError("Falta POLYGON_API_KEY")

# ---------------------------
# Configuración filtros opciones
# ---------------------------
MIN_DTE = 10
MAX_DTE = 55

MIN_DELTA = 0.25
MAX_DELTA = 0.35

CONTRACT_TYPE = "call"

# ---------------------------
# Lista de tickers (mock / input manual)
# ---------------------------
tickers_validos = ["QCOM"]  # podés poner varios

print(f"Procesando {len(tickers_validos)} tickers...\n")

# ---------------------------
# Consulta Polygon
# ---------------------------
def buscar_opciones_por_ticker(ticker_raiz):
    hoy = datetime.now()
    url = f"https://api.polygon.io/v3/snapshot/options/{ticker_raiz}?limit=250&apiKey={POLYGON_API_KEY}"

    resultados = []
    buscando = True

    while url and buscando:
        r = requests.get(url)
        if r.status_code != 200:
            print(f"❌ Error Polygon {ticker_raiz}: {r.status_code}")
            break

        data = r.json()
        contracts = data.get("results", [])

        for c in contracts:
            details = c.get("details", {})
            greeks = c.get("greeks", {})
            day = c.get("day", {})

            if details.get("contract_type") != CONTRACT_TYPE:
                continue

            expiry = details.get("expiration_date")
            if not expiry:
                continue

            fecha_vto = datetime.strptime(expiry, "%Y-%m-%d")
            dte = (fecha_vto - hoy).days

            # Corte temprano (Polygon viene ordenado por vto)
            if dte > MAX_DTE:
                buscando = False
                break

            delta = greeks.get("delta")
            if delta is None or not (MIN_DELTA <= delta <= MAX_DELTA):
                continue

            if MIN_DTE <= dte <= MAX_DTE:
                resultados.append({
                    "opcion": details.get("ticker"),
                    "strike": details.get("strike_price"),
                    "vto": expiry,
                    "dte": dte,
                    "delta": round(delta, 4),
                    "iv": c.get("implied_volatility"),
                    "oi": c.get("open_interest", 0),
                    "volume": day.get("volume", 0),
                    "close": day.get("close")
                })

        next_url = data.get("next_url")
        url = f"{next_url}&apiKey={POLYGON_API_KEY}" if next_url else None

    return resultados

# ---------------------------
# Ejecutar y mostrar
# ---------------------------
total_opciones = 0

for ticker in tickers_validos:
    print(f"📌 TICKER: {ticker}")
    print("=" * 80)

    opciones = buscar_opciones_por_ticker(ticker)

    if not opciones:
        print("No se encontraron PUTs válidos\n")
        continue

    for o in opciones:
        print(
            f"{o['opcion']:20} | "
            f"Vto {o['vto']} | "
            f"Strk {o['strike']:>6} | "
            f"DTE {o['dte']:>3} | "
            f"Δ {o['delta']:>6} | "
            f"IV {o['iv']:.3f} | "
            f"OI {o['oi']:>5} | "
            f"Vol {o['volume']:>5}"
        )

    print(f"\n➡️ Total opciones válidas: {len(opciones)}\n")
    total_opciones += len(opciones)

    time.sleep(0.25)

print(f"✅ Proceso finalizado. Total opciones encontradas: {total_opciones}")


Procesando 1 tickers...

📌 TICKER: QCOM
O:QCOM260501C00138000 | Vto 2026-05-01 | Strk    138 | DTE  16 | Δ 0.3482 | IV 0.468 | OI   101 | Vol     2
O:QCOM260501C00139000 | Vto 2026-05-01 | Strk    139 | DTE  16 | Δ 0.3294 | IV 0.483 | OI     8 | Vol     2
O:QCOM260501C00140000 | Vto 2026-05-01 | Strk    140 | DTE  16 | Δ 0.2993 | IV 0.470 | OI   451 | Vol     8
O:QCOM260501C00141000 | Vto 2026-05-01 | Strk    141 | DTE  16 | Δ 0.2761 | IV 0.473 | OI    12 | Vol     1
O:QCOM260501C00142000 | Vto 2026-05-01 | Strk    142 | DTE  16 | Δ   0.25 | IV 0.471 | OI    77 | Vol     1
O:QCOM260501C00185000 | Vto 2026-05-01 | Strk    185 | DTE  16 | Δ  0.304 | IV 2.121 | OI     0 | Vol     0
O:QCOM260501C00190000 | Vto 2026-05-01 | Strk    190 | DTE  16 | Δ  0.298 | IV 2.197 | OI     0 | Vol     0
O:QCOM260501C00200000 | Vto 2026-05-01 | Strk    200 | DTE  16 | Δ 0.2869 | IV 2.339 | OI     0 | Vol     0
O:QCOM260501C00205000 | Vto 2026-05-01 | Strk    205 | DTE  16 | Δ 0.2809 | IV 2.400 | OI     0 